In [56]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul2-train/1_2_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
x_train = x_train.astype(np.float32)
print(x_train.shape, x_train.dtype, y_train.shape, y_train.dtype)

test_data = np.load(file='Numarul2-test/1_2_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
x_test = x_test.astype(np.float32)

N_CLASSES = len(np.unique(ar=y_train))
print(f"Numarul de clase este: {N_CLASSES}")

X_train = torch.from_numpy(x_train)
X_test = torch.from_numpy(x_test)

mean = torch.mean(input=X_train, dim=0)
std = torch.std(input=X_train, dim=0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

n = X_train_norm.shape[0]
K = (X_train_norm.T @ X_train_norm) / (n - 1)
D, V = torch.linalg.eigh(K)

# DP trebuie sa fie 1
print(K)

idx = torch.argsort(D, descending=True)
D = D[idx]
V = V[:, idx]

FEAT_NUM = int(X_train_norm.shape[1] / 4)
print(f"Numarul initial de trasaturi initial este: {X_train_norm.shape[1]} asa ca un sfert din ele este {FEAT_NUM}")

x_train= X_train_norm @ V[:, :FEAT_NUM]
x_test = X_test_norm @ V[:, :FEAT_NUM]


(35000, 1024) float32 (35000,) int64
Numarul de clase este: 7
tensor([[1.0000, 0.9774, 0.9474,  ..., 0.3077, 0.3200, 0.3378],
        [0.9774, 1.0000, 0.9785,  ..., 0.3039, 0.3158, 0.3301],
        [0.9474, 0.9785, 1.0000,  ..., 0.3012, 0.3120, 0.3247],
        ...,
        [0.3077, 0.3039, 0.3012,  ..., 1.0000, 0.9763, 0.9356],
        [0.3200, 0.3158, 0.3120,  ..., 0.9763, 1.0000, 0.9736],
        [0.3378, 0.3301, 0.3247,  ..., 0.9356, 0.9736, 1.0000]])
Numarul initial de trasaturi initial este: 1024 asa ca un sfert din ele este 256


In [57]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.feature_vectors = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature_vector = self.feature_vectors[idx]
        label = self.labels[idx]

        feature_vector = torch.as_tensor(feature_vector, dtype=torch.float32)
        label = torch.as_tensor(label, dtype=torch.int64)

        return feature_vector, label

train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [58]:
class Model(nn.Module):
    def __init__(self, in_features=FEAT_NUM, hidden=128, n_classes=N_CLASSES, act=nn.functional.relu):
        super().__init__()
        self.act = act

        self.fc1 = nn.Linear(in_features=in_features, out_features=in_features)
        self.drop1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(in_features=in_features, out_features=hidden)
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(in_features=hidden, out_features=n_classes)


    def forward(self, x):
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 50


In [59]:
def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total

best = 0
for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"train_loss: {train_loss:.4f} || train_acc: {train_acc:.4f} || test_loss: {test_loss:.4f} || test_acc: {test_acc:.4f}")
    if test_acc > best:
        best = test_acc
        torch.save(model, 'model.pt')

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')

  2%|▏         | 1/50 [00:01<00:52,  1.06s/it]

train_loss: 1.5642 || train_acc: 0.3926 || test_loss: 1.4473 || test_acc: 0.4477


  4%|▍         | 2/50 [00:02<00:49,  1.03s/it]

train_loss: 1.3868 || train_acc: 0.4702 || test_loss: 1.3489 || test_acc: 0.4826


  6%|▌         | 3/50 [00:03<00:47,  1.01s/it]

train_loss: 1.2907 || train_acc: 0.5081 || test_loss: 1.3095 || test_acc: 0.4963


... (90 outputs left out)

 98%|█████████▊| 49/50 [00:44<00:00,  1.22it/s]

train_loss: 0.5786 || train_acc: 0.7882 || test_loss: 1.4956 || test_acc: 0.5336


100%|██████████| 50/50 [00:45<00:00,  1.10it/s]

train_loss: 0.5763 || train_acc: 0.7879 || test_loss: 1.5125 || test_acc: 0.5334
0.5480
